# 01. Descarga de archivos desde S3

## Objetivo
 Descargar archivos (CSV) desde un repositorio en **Amazon S3**, permitiendo parametrizar:
- El repositorio (bucket/prefix) (`repo`)
- El nombre del archivo (`filename`)
- La ruta de salida local (`out_path`)
- El modo de autenticación: **anónimo** (bucket público) o **con credenciales** (bucket privado)

## Enfoque
Se implementó una función `download_s3_file(...)` que:
1. Interpreta el parámetro `repo`, soportando formatos:
   - `s3://<bucket>/<prefix-opcional>`
   - `https://<bucket>.s3.amazonaws.com/<prefix-opcional>`
2. Construye el `key` del objeto combinando `prefix` + `filename`.
3. Descarga el archivo con `boto3`:
   - `anon=True`: requests **unsigned** (sin credenciales), ideal para buckets públicos.
   - `anon=False`: requests **signed** usando credenciales configuradas (variables de entorno, `.env` o AWS profile).

## Consideraciones / Riesgos
- Si se usa `anon=False` sin credenciales configuradas, la descarga falla con `NoCredentialsError`.
- En algunos buckets públicos, las políticas pueden permitir solo acceso anónimo (unsigned) y rechazar requests firmadas con 403.
- Por seguridad, las credenciales no deben quedar hardcodeadas ni commiteadas (usar `.env` + `.gitignore`).


# Librerias

In [31]:
import os
from urllib.parse import urlparse

import boto3
from botocore import UNSIGNED
from botocore.client import Config
from botocore.exceptions import NoCredentialsError, PartialCredentialsError, ClientError

In [6]:
def download_s3_file(
    repo: str,
    filename: str,
    out_path: str,
    anon: bool = False,
    profile: str | None = None,
) -> None:
   #Validamos los parametros
    if not repo or not isinstance(repo, str):
        raise ValueError("El nombre del repositorio no puede ser vacio")
    if not filename or not isinstance(filename, str):
        raise ValueError("El nombre de archivo no puede ser vacio")
    if not out_path or not isinstance(out_path, str):
        raise ValueError("La ruta no puede ser vacia")

    u = urlparse(repo)
    #Validamos que el repos tenga formato correcto
    if repo.startswith("s3://"):
        bucket = u.netloc
        prefix = u.path.lstrip("/")
    elif repo.startswith("http://") or repo.startswith("https://"):
        bucket = u.netloc.split(".")[0]
        prefix = u.path.lstrip("/")
    else:
        raise ValueError("Repositoria debe comenzar con s3:// or http(s)://")

    if not bucket:
        raise ValueError(f"NO se pudo parsear el nombre de repositorio: {repo}")

    key = f"{prefix}/{filename}".strip("/") if prefix else filename

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

   
    try:
        if anon:
            s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
        else:
            session = boto3.Session(profile_name=profile) if profile else boto3.Session()
            s3 = session.client("s3")
    except Exception as e:
        raise RuntimeError(f"Failed to create S3 client: {e}") from e

   
    try:
        s3.download_file(bucket, key, out_path)
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Local path not found or invalid: {out_path}") from e
    except (NoCredentialsError, PartialCredentialsError) as e:
        raise RuntimeError(
            "No se encontraron credenciales de AWS. "

        ) from e
    except ClientError as e:
        code = e.response.get("Error", {}).get("Code", "Unknown")
        msg = e.response.get("Error", {}).get("Message", str(e))

        if code in ("404", "NoSuchKey", "NotFound"):
            raise FileNotFoundError(f"S3 object not found: s3://{bucket}/{key}") from e
        if code in ("403", "AccessDenied"):
            raise PermissionError(
                f"Acceso denegado para s3://{bucket}/{key}. "
                f"{'Intenta anon=True (bucket publico)' if not anon else 'Bucket no publico.'}"
            ) from e

        raise RuntimeError(f"S3 ClientError ({code}): {msg} | s3://{bucket}/{key}") from e

    print(f"Descargado: s3://{bucket}/{key} -> {out_path}")

In [41]:
download_s3_file("https://desafio-rkd.s3.amazonaws.com", "netflix_titles.csv", "data/netflix_titles.csv", anon=True)
download_s3_file("https://desafio-rkd.s3.amazonaws.com", "disney_plus_titles.csv", "data/disney_plus_titles.csv", anon=True)
print("OK")

Descargado: s3://desafio-rkd/netflix_titles.csv -> data/netflix_titles.csv
Descargado: s3://desafio-rkd/disney_plus_titles.csv -> data/disney_plus_titles.csv
OK
